In [ ]:
include("LopezStability.jl")
include("CRD_STA.jl")
include("LopezBaseflow.jl")
include("NeutralContinuation.jl")
using .LopezBaseflow
using .NeutralContinuation
using Plots
using LinearAlgebra
using NonlinearEigenproblems
using DelimitedFiles
using BSplineKit

In [ ]:
function eigsol(F,G,H,T,R,omega,be,N_cheb,D,D2,c,num; v0=nothing)
    eigval, eigvec = LopezStability.eigsol_lopez(
        F, G, H, T,
        R, omega, be,
        N_cheb, D, D2,
        c, num; initial_vector=v0,
    )
    return eigval,eigvec
end

In [ ]:
function ODE(N_cheb,Ro; domain=:rational, ymax=40.0)
    Co = 2 - Ro - Ro^2
    u0,v0,w0,f,q,D,D2,x = baseflow_var(N_cheb,Ro,Co; domain=domain, ymax=ymax)
    return u0,v0,w0,f,q,D,D2,x
end
function BF(N_cheb,Tw,Mr,u0,v0,w0,f,q,D,D2,x)
    gamma = 1.4
    sigma = 0.72
    H,T = T_ca(Mr,f,q,w0,gamma,Tw)
    F,G,H,T,rho,z = interp(u0,v0,H,T,x,N_cheb,"sim")
    lam = - (2/3) * T
    kappa = (1/sigma) * T
    return F,G,H,T,rho,z,lam,kappa,D,D2
end
function eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,c,num;
                v0=nothing, regularization=0.0, balance=true, return_info=false,
                property_perturbations::Bool=true, base_property_variation::Bool=true)
    result = solve_spatial_mode(
        F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,
        N_cheb,Ro,Co,D,D2;
        target=c, neigs=min(num,2), initial_vector=v0,
        regularization=regularization, balance=balance,
        maxit=800, tol=1e-10,
        property_perturbations=property_perturbations,
        base_property_variation=base_property_variation,
    )
    return return_info ? result : (result.values, result.vectors)
end

In [ ]:
function interp(u0,v0,w0,T0,x,N)
    F = zeros(N+1)
    G = zeros(N+1)
    H = zeros(N+1)
    T = zeros(N+1)

    z = range(0, 40, 2001)

    itu = BSplineKit.interpolate(z, u0, BSplineOrder(4))
    itv = BSplineKit.interpolate(z, v0, BSplineOrder(4))
    itw = BSplineKit.interpolate(z, w0, BSplineOrder(4))
    itt = BSplineKit.interpolate(z, T0, BSplineOrder(4))

    for i in 1:N+1
        F[i] = itu(x[i])
        G[i] = itv(x[i])
        H[i] = itw(x[i])
        T[i] = itt(x[i])
    end

    return F, G, H, T
end

In [ ]:
N_cheb = 69
Ro = -1.0
Tw = 1.0
Mr = 0.3
gamma = 1.4
sigma = 0.72
Co = 2-Ro-Ro^2
be = 0.07759
num = 1
omega = 0.0
R = 280
c = 0.4
Ma = Mr/R
u0,v0,w0,f,q,D,D2,x = ODE(N_cheb,Ro)
F,G,H,T,rho,z,lam,kappa,D,D2 = BF(N_cheb,Tw,Mr,u0,v0,w0,f,q,D,D2,x)

In [ ]:
eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,N_cheb,Ro,Co,D,D2,c,num;
                v0=nothing, regularization=0.0, balance=true, return_info=false)

In [ ]:
plot(z,F,xlims=[0,20])
plot!(z,G)
plot!(z,H)
plot!(z,T)

In [ ]:
R = 440.88
omega = 0.0
be = 0.04672
Tw = 1.0
alpha_ref = 0.13
z, H, F, G, T, dF, dG, dT, info = LopezBaseflow.get_baseflow(Tw)
N_cheb = 99
c_ini = 0.3
num = 3
D,D2,x = CRD_BF.Cheb(N_cheb)
F,G,H,T = interp(F,G,H,T,x,N_cheb)

In [ ]:
eigval, eigvec = LopezStability.eigsol_lopez(
    F, G, H, T,
    R, omega, be,
    N_cheb, D, D2,
    c_ini, num,
)
@show eigval

In [ ]:
plot(x,F,xlims=[0,20])
plot!(x,G)
plot!(x,H)
plot!(x,T)

In [ ]:
function cur_legacy(Tw,omega,R_ini,c_ini,be_ini,num;
                    model=:lopez, Mr=0.3, Ro=-1.0, N_cheb=nothing,
                    property_perturbations::Bool=true, base_property_variation::Bool=true,
                    min_mode_overlap::Float64=0.60, continuation::Symbol=:optimized,
                    beta_step::Float64=0.0008, neutral_tol::Float64=1.0e-7,
                    R_tol::Float64=1.0e-4, corrector_R_step::Float64=1.0,
                    max_scan_steps::Int=80, max_prediction_step::Float64=30.0,
                    max_curve_points::Union{Nothing,Int}=nothing,
                    adaptive_beta_refinement::Bool=true,
                    min_beta_step::Union{Nothing,Float64}=nothing,
                    fallback_to_legacy::Bool=false,
                    max_legacy_no_progress::Int=1, keep_logs::Bool=false)
    model = Symbol(lowercase(String(model)))
    model = model in (:full, :crd, :fully_compressible) ? :compressible : model
    model in (:lopez, :compressible) ||
        throw(ArgumentError("model must be :lopez or :compressible"))
    N_cheb = isnothing(N_cheb) ? (model === :lopez ? 69 : 69) : Int(N_cheb)
    N_cheb >= 20 || throw(ArgumentError("N_cheb must be at least 20"))
    num = clamp(Int(num),1,2)
    0.0 <= min_mode_overlap <= 1.0 ||
        throw(ArgumentError("min_mode_overlap must lie in [0,1]"))
    continuation in (:optimized,:legacy_scan) ||
        throw(ArgumentError("continuation must be :optimized or :legacy_scan"))
    beta_step > 0 || throw(ArgumentError("beta_step must be positive"))
    min_beta_step = isnothing(min_beta_step) ? beta_step/16 : Float64(min_beta_step)
    0 < min_beta_step <= beta_step ||
        throw(ArgumentError("min_beta_step must lie in (0,beta_step]"))
    neutral_tol > 0 || throw(ArgumentError("neutral_tol must be positive"))
    R_tol > 0 || throw(ArgumentError("R_tol must be positive"))
    corrector_R_step > 0 || throw(ArgumentError("corrector_R_step must be positive"))
    max_scan_steps >= 1 || throw(ArgumentError("max_scan_steps must be positive"))
    max_prediction_step > 0 || throw(ArgumentError("max_prediction_step must be positive"))
    max_curve_points === nothing || max_curve_points >= 1 ||
        throw(ArgumentError("max_curve_points must be positive"))
    max_legacy_no_progress >= 1 ||
        throw(ArgumentError("max_legacy_no_progress must be positive"))
    R_ini > 0 || throw(ArgumentError("R_ini must be positive"))
    model === :compressible && Mr <= 0 &&
        throw(ArgumentError("Mr must be positive for the compressible model"))

    be_step = -0.0005
    DM,D2M,x = CRD_BF.Cheb(N_cheb)
    if model === :lopez
        z, H, F, G, T, dF, dG, dT, info = LopezBaseflow.get_baseflow(Tw)
        F,G,H,T = interp(F,G,H,T,x,N_cheb)
    else
        gamma = 1.4
        sigma = 0.72
        Co = 2 - Ro - Ro^2
        u0,v0,w0,f,q,DM,D2M,x = ODE(N_cheb,Ro)
        F,G,H,T,rho,z,lam,kappa,DM,D2M =
            BF(N_cheb,Tw,Mr,u0,v0,w0,f,q,DM,D2M,x)
    end

    function solve_at(R, be, target, nvalues; v0=nothing)
        if model === :lopez
            return eigsol(F,G,H,T,R,omega,be,N_cheb,DM,D2M,target,nvalues; v0=v0)
        end
        R > 0 || throw(ArgumentError("R must remain positive for Ma = Mr/R"))
        Ma = Mr / R
        return eigsol(F,G,H,rho,lam,kappa,T,sigma,gamma,R,Ma,omega,be,
                      N_cheb,Ro,Co,DM,D2M,target,nvalues; v0=v0,
                      property_perturbations=property_perturbations,
                      base_property_variation=base_property_variation)
    end

    function continue_modes(R, be, previous_values, previous_vectors)
        return NeutralContinuation.continue_modes(
            solve_at,R,be,previous_values,previous_vectors;
            min_overlap=min_mode_overlap,
        )
    end

    property_tag = property_perturbations ? "on" : "off"
    base_property_tag = base_property_variation ? "variable" : "frozen"
    model_tag = model === :lopez ? "model=lopez" : "model=compressible_Mr=$(Mr)"
    case_tag = "$(model_tag)_propPert=$(property_tag)_baseProp=$(base_property_tag)"
    if model === :lopez && (!property_perturbations || !base_property_variation)
        @warn "Material-property switches are recorded in filenames but apply only to the compressible operator"
    end
    output_file = "output_ome=$(omega)_Tw=$(Tw)_$(case_tag).dat"
    eig_output_file = "output_eig_ome=$(omega)_Tw=$(Tw)_$(case_tag).dat"
    curve_file = "ome=$(omega)_Tw=$(Tw)_$(case_tag).dat"
    function cleanup_logs()
        for path in (output_file,eig_output_file)
            isfile(path) && rm(path; force=true)
        end
    end
    function write_curve_checkpoint(data)
        variables = "Variables=\"omega\" \"R\" \"beta\" \"alpha_r_1\" \"alpha_i_1\" \"alpha_r_2\" \"alpha_i_2\""
        zone = "Zone T=\"omega=$(omega),Tw=$(Tw),$(case_tag)\""
        open(curve_file,"w") do io
            println(io,variables)
            println(io,zone)
            size(data,1) > 1 && writedlm(io,data[2:end,:])
        end
    end
    initial = []
    tempvec_1 = [0 0 0 0 0 0 0]
    writedlm(output_file,initial)
    writedlm(eig_output_file,initial)
    eigval_ori,eigvec = solve_at(R_ini,be_ini,c_ini,num)
    eigval = eigval_ori
    open(eig_output_file, "a") do io
        write(io,"be=$be_ini,eig=$eigval_ori\n")
    end
    # eigval = sort(eigval_ori, by=real)
    if minimum(abs.(imag.(eigval_ori))) < 3e-5
        initial = reshape([
            omega,R_ini,be_ini,real(eigval_ori[1]),imag(eigval_ori[1]),
            real(eigval_ori[end]),imag(eigval_ori[end]),
        ],1,:)
    elseif imag(eigval_ori[1]) < 0
        for be = be_ini :  be_step : -0.5
            sig_last1 = sign(imag(eigval[1]))
            sig_last2 = sign(imag(eigval[end]))
            eigval,eigvec,overlaps = continue_modes(R_ini,be,eigval,eigvec)
            sig_now1 = sign(imag(eigval[1]))
            sig_now2 = sign(imag(eigval[end]))
            # point = filter(x -> abs(imag(x)) < 0.0004 , eigval)
            open(eig_output_file, "a") do io
                write(io,"be=$be,eig=$eigval,overlap=$overlaps\n")
            end
            if sig_last1 * sig_now1 < 0 || sig_last2 * sig_now2 < 0
                initial = reshape([omega,R_ini,be,real(eigval[1]),imag(eigval[1]),real(eigval[end]),imag(eigval[end])],1,:)
                break
            end
        end
    elseif imag(eigval_ori[1]) > 0
        for be = be_ini : - be_step : 0.5
           sig_last1 = sign(imag(eigval[1]))
            sig_last2 = sign(imag(eigval[end]))
            eigval,eigvec,overlaps = continue_modes(R_ini,be,eigval,eigvec)
            sig_now1 = sign(imag(eigval[1]))
            sig_now2 = sign(imag(eigval[end]))
            # point = filter(x -> abs(imag(x)) < 0.0004 , eigval)
            open(eig_output_file, "a") do io
                write(io,"be=$be,eig=$eigval,overlap=$overlaps\n")
            end
            if sig_last1 * sig_now1 < 0 || sig_last2 * sig_now2 < 0
                initial = reshape([omega,R_ini,be,real(eigval[1]),imag(eigval[1]),real(eigval[end]),imag(eigval[end])],1,:)
                break
            end
        end
    end
    isempty(initial) && error("No neutral crossing found in the initial beta scan for model=$(model)")
    total = initial
    be = initial[3] - be_step
    dir = 0
    boundlen = 3
 # CACULATE

    if continuation === :optimized
        current_beta_step = beta_step
        be = initial[3] + current_beta_step
        fallback_requested = false

        open(output_file,"a") do trace_io
            while true
                previous_R = total[end,2]
                predicted_delta_R = 0.0
                if size(total,1) >= 2
                    previous_delta_beta = total[end,3] - total[end-1,3]
                    if abs(previous_delta_beta) > eps(Float64)
                        slope = (total[end,2] - total[end-1,2]) / previous_delta_beta
                        predicted_delta_R = slope * (be - total[end,3])
                    end
                end
                predicted_delta_R = clamp(
                    predicted_delta_R,-max_prediction_step,max_prediction_step,
                )
                R_guess = clamp(previous_R + predicted_delta_R,1.0e-5,699.999)
                preferred_direction = predicted_delta_R > 0 ? 1 :
                    (predicted_delta_R < 0 ? -1 : 0)
                active_index = argmin(abs.(imag.(eigval)))
                evaluation_count = 0
                on_evaluation = state -> begin
                    evaluation_count += 1
                    write(
                        trace_io,
                        "R=$(state.R),beta=$be,eig=$(state.values)," *
                        "mode=corrector,eval=$evaluation_count," *
                        "overlap=$(state.overlaps),residual=$(state.residual)\n",
                    )
                end

                root = try
                    NeutralContinuation.find_neutral_R(
                        solve_at,be,R_guess,eigval,eigvec,active_index;
                        R_step=corrector_R_step,
                        preferred_direction=preferred_direction,
                        max_scan_steps=max_scan_steps,max_refine=30,
                        neutral_tol=neutral_tol,R_tol=R_tol,
                        R_bounds=(1.0e-6,700.0),
                        max_R_deviation=max(40.0,2abs(predicted_delta_R)+10.0),
                        min_overlap=min_mode_overlap,
                        on_evaluation=on_evaluation,
                    )
                catch exception
                    exception isa InterruptException && rethrow()
                    reduced_step = NeutralContinuation.refined_beta_step(
                        current_beta_step,min_beta_step;
                        enabled=adaptive_beta_refinement,
                    )
                    if reduced_step !== nothing
                        old_step = current_beta_step
                        current_beta_step = reduced_step
                        write(
                            trace_io,
                            "beta_retry=$(be),last_beta=$(total[end,3])," *
                            "old_step=$(old_step),new_step=$(current_beta_step)," *
                            "reason=$(sprint(showerror,exception))\n",
                        )
                        flush(trace_io)
                        @warn "Neutral corrector could not bracket a root; reducing beta step" failed_beta=be last_beta=total[end,3] old_step new_step=current_beta_step
                        be = total[end,3] + current_beta_step
                        continue
                    end
                    fallback_requested = fallback_to_legacy
                    write(
                        trace_io,
                        "curve_stop_beta=$(be),last_beta=$(total[end,3])," *
                        "last_R=$(total[end,2]),beta_step=$(current_beta_step)," *
                        "reason=$(sprint(showerror,exception))\n",
                    )
                    flush(trace_io)
                    if fallback_requested
                        @warn "Optimized neutral corrector stopped; explicitly falling back to legacy scan" be R_guess exception
                    else
                        @warn "Neutral curve ended because no root could be bracketed at the minimum beta step" failed_beta=be last_beta=total[end,3] last_R=total[end,2] beta_step=current_beta_step exception
                    end
                    break
                end

                abs(root.residual) <= neutral_tol || error(
                    "Accepted neutral residual $(root.residual) exceeds $neutral_tol",
                )
                eigval,eigvec,overlaps = root.values,root.vectors,root.overlaps
                row = reshape([
                    omega,root.R,be,real(eigval[1]),imag(eigval[1]),
                    real(eigval[end]),imag(eigval[end]),
                ],1,:)
                total = [total;row]
                write(
                    trace_io,
                    "neutral_R=$(root.R),beta=$be,eig=$eigval," *
                    "scan_iterations=$(root.scan_iterations)," *
                    "refine_iterations=$(root.refine_iterations)," *
                    "evaluations=$evaluation_count,overlap=$overlaps\n",
                )
                flush(trace_io)
                write_curve_checkpoint(total)

                curve_points = size(total,1) - 1
                max_curve_points !== nothing && curve_points >= max_curve_points && break
                total[end,2] > 500 && size(total,1) > 30 && break
                be += current_beta_step
            end
        end
        if !fallback_requested
            write_curve_checkpoint(total)
            keep_logs || cleanup_logs()
            return total
        end
    end

    legacy_no_progress = 0
    while true
        rows_before_iteration = size(total,1)
        index = findall(x->x==findmin([total[end,5],total[end,7]])[1],total[end,:])
        c = total[end,index[1] - 1]
        eigval,eigvec,overlaps = continue_modes(total[end,2],be,eigval,eigvec)
        center_values = copy(eigval)
        center_vectors = copy(eigvec)
        eigval_1,eigvec_1,overlaps_1 = continue_modes(
            total[end,2],be-0.001,center_values,center_vectors,
        )
        eigval_2,eigvec_2,overlaps_2 = continue_modes(
            total[end,2],be+0.001,center_values,center_vectors,
        )
        tracked_index = argmin(abs.(imag.(center_values)))
        index1 = tracked_index
        index2 = tracked_index
        num = 2
        if size(total,1) > 3 && abs(total[end,2] - total[end-1,2]) <=2
            R_step = 0.25
        else
            R_step = 0.5
        end
        if (imag(eigval_1[index1]) < 0 && imag(eigval_2[index2]) > 0) || (imag(eigval_1[index1]) > 0 && imag(eigval_2[index2]) > 0) || dir == -1
            mode = 1
        elseif (imag(eigval_1[index1]) > 0 && imag(eigval_2[index2]) < 0) || (imag(eigval_1[index1]) < 0 && imag(eigval_2[index2]) < 0)
            mode = 2
        end
        
        if mode == 1 

            scan_values = copy(center_values)
            scan_vectors = copy(center_vectors)
            for R = total[end,2] : R_step : 700

                eigval,eigvec,overlaps = continue_modes(
                    R,be,scan_values,scan_vectors,
                )
                scan_values,scan_vectors = eigval,eigvec

                tempvec_1 = [tempvec_1;[omega R be real(eigval[1]) imag(eigval[1]) real(eigval[end] ) imag(eigval[end])]]     
                len = size(tempvec_1,1)
                open(output_file, "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len,overlap=$overlaps\n")
                end

                if ((tempvec_1[end-1, 5] * tempvec_1[end,5]) < 0 && abs(tempvec_1[end,5]) < 0.001) || (abs(tempvec_1[end,5])<3e-5) || ((tempvec_1[end-1, 7] * tempvec_1[end,7]) < 0) || (abs(tempvec_1[end,7])<3e-5) 

                    total = [total ; tempvec_1[end:end,:]]
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end

                if (len > boundlen && abs(tempvec_1[end,5]) > abs(tempvec_1[end-1,5]) && abs(tempvec_1[end,7]) > abs(tempvec_1[end-1,7]))

                    mode = 2
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
            end        
        end


        if mode == 2

            scan_values = copy(center_values)
            scan_vectors = copy(center_vectors)
            for R = total[end,2]: -R_step : 0
                
                eigval,eigvec,overlaps = continue_modes(
                    R,be,scan_values,scan_vectors,
                )
                scan_values,scan_vectors = eigval,eigvec
                tempvec_1 = [tempvec_1;[omega R be real(eigval[1]) imag(eigval[1]) real(eigval[end] ) imag(eigval[end])]]     
                len = size(tempvec_1,1)
                open(output_file, "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len,overlap=$overlaps\n")
                end
                if ((tempvec_1[end-1, 5] * tempvec_1[end,5]) < 0 && abs(tempvec_1[end,5]) < 0.001) || (abs(tempvec_1[end,5])<3e-5) || ((tempvec_1[end-1, 7] * tempvec_1[end,7]) < 0) || (abs(tempvec_1[end,7])<3e-5) 

                    total = [total ; tempvec_1[end:end,:]]
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
                
                if (len > boundlen && abs(tempvec_1[end,5]) > abs(tempvec_1[end-1,5]) && abs(tempvec_1[end,7]) > abs(tempvec_1[end-1,7]))

                    mode = 1
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
            end        
        end

        if mode == 1

            scan_values = copy(center_values)
            scan_vectors = copy(center_vectors)
            for R = total[end,2]: R_step : 700

                if total[end,3] == be

                    break

                end 
                
                eigval,eigvec,overlaps = continue_modes(
                    R,be,scan_values,scan_vectors,
                )
                scan_values,scan_vectors = eigval,eigvec
                tempvec_1 = [tempvec_1;[omega R be real(eigval[1]) imag(eigval[1]) real(eigval[end] ) imag(eigval[end])]]     

                len = size(tempvec_1,1)
                open(output_file, "a") do io
                    write(io, "R=$R,beta=$be,eig=$eigval,mode=$mode,$len,overlap=$overlaps\n")
                end

                if ((tempvec_1[end-1, 5] * tempvec_1[end,5]) < 0 && abs(tempvec_1[end,5]) < 0.001) || (abs(tempvec_1[end,5])<3e-5) || ((tempvec_1[end-1, 7] * tempvec_1[end,7]) < 0) || (abs(tempvec_1[end,7])<3e-5) 

                    total = [total ; tempvec_1[end:end,:]]
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
                
                if (len > boundlen && abs(tempvec_1[end,5]) > abs(tempvec_1[end-1,5]) && abs(tempvec_1[end,7]) > abs(tempvec_1[end-1,7]))

                    mode = 2
                    tempvec_1 = [0 0 0 0 0 0 0]

                    break
                end
            end        
        end
        if size(total,1) == rows_before_iteration
            legacy_no_progress += 1
            @warn "Legacy scan found no neutral point" beta=be last_R=total[end,2] no_progress=legacy_no_progress
            if legacy_no_progress >= max_legacy_no_progress
                @warn "Stopping legacy continuation to prevent an infinite no-progress loop" beta=be last_R=total[end,2]
                break
            end
        else
            legacy_no_progress = 0
        end
        c = total[end,4]
        be += beta_step
        write_curve_checkpoint(total)
        if total[end,2] > 500 && size(total,1) > 30 
            break
        end
    end
    write_curve_checkpoint(total)
    keep_logs || cleanup_logs()
    return total
end

In [ ]:
for Tw = 1.06 : 0.02 : 1.2
    R_ini = 500.0
    omega = 0.0
    beta_ini = 0.04
    alpha_target = 0.1
    num = 1
    curve_crd = cur_legacy(
        Tw, omega, R_ini, alpha_target, beta_ini, num;
        model=:lopez,
        Mr=0.3,
        Ro=-1.0,
        continuation=:optimized,
        N_cheb=69,
        beta_step=0.0008,
        neutral_tol=1e-7,
        property_perturbations=true,
        base_property_variation=true,
    )
    curve_crd = cur_legacy(
        Tw, omega, R_ini, alpha_target, beta_ini, num;
        model=:compressible,
        Mr=0.3,
        Ro=-1.0,
        continuation=:optimized,
        N_cheb=69,
        beta_step=0.0008,
        neutral_tol=1e-7,
        property_perturbations=true,
        base_property_variation=true,
    )
    curve_crd = cur_legacy(
        Tw, omega, R_ini, alpha_target, beta_ini, num;
        model=:compressible,
        Mr=0.3,
        Ro=-1.0,
        continuation=:optimized,
        N_cheb=69,
        beta_step=0.0008,
        neutral_tol=1e-7,
        property_perturbations=false,
        base_property_variation=true,
    )
    curve_crd = cur_legacy(
        Tw, omega, R_ini, alpha_target, beta_ini, num;
        model=:compressible,
        Mr=0.3,
        Ro=-1.0,
        continuation=:optimized,
        N_cheb=69,
        beta_step=0.0008,
        neutral_tol=1e-7,
        property_perturbations=false,
        base_property_variation=false,
    )
end

## Sutherland base flow at specified Tw, Mr, and R

Set `Tw_suth`, `Mr_suth`, and `R_suth` below, then run the next cells to generate and visualize the Sutherland-viscosity base-flow profile at that radial position.  Set `save_outputs_suth = true` only when CSV and metadata files are needed.


In [ ]:
using Printf
using Plots

# User parameters for the Sutherland marching solver.
Tw_suth = 1.4
Mr_suth = 0.3
R_suth = 200.0

# Numerical controls. Increase Nr/Nz for final data or grid-convergence checks.
r0_suth = 1.0
Nr_suth = 41
Nz_suth = 81
zmax_suth = 20.0
newton_tol_suth = 1.0e-8
newton_maxiter_suth = 14

# Set this to true only when CSV/metadata files are needed.
save_outputs_suth = false

out_dir_suth = @sprintf("sutherland_Tw%.3g_Mr%.3g_R%.3g", Tw_suth, Mr_suth, R_suth)
Minf_suth = Mr_suth / R_suth

nothing


In [ ]:
# Run the Sutherland solver inside the notebook.
include("SutherlandMarching.jl")
Suth = SutherlandMarching

p_suth = Suth.Params(
    Tw = Tw_suth,
    Minf = Minf_suth,
    r0 = r0_suth,
    rmax = R_suth,
    Nr = Nr_suth,
    zmax = zmax_suth,
    Nz = Nz_suth,
    newton_tol = newton_tol_suth,
    newton_maxiter = newton_maxiter_suth,
    out_dir = out_dir_suth,
    verbose = false,
)

r_values_suth, z_suth, stations_suth = Suth.solve_marching(p_suth)

profile_all_suth = nothing
profile_R_suth = nothing
summary_suth = nothing
metadata_suth = nothing
if save_outputs_suth
    profile_all_suth, profile_R_suth, summary_suth, metadata_suth = Suth.write_outputs(r_values_suth, z_suth, stations_suth, p_suth)
end

st_R_suth = stations_suth[end]
U_suth = st_R_suth.U
V_suth = st_R_suth.V
W_suth = st_R_suth.W
T_suth = st_R_suth.T
rho_suth = Suth.suth_rho.(T_suth)
mu_suth = [Suth.suth_mu(Tj, p_suth) for Tj in T_suth]
kappa_suth = mu_suth ./ p_suth.sigma
u_physical_suth = R_suth .* U_suth
v_rotating_physical_suth = R_suth .* V_suth
v_inertial_physical_suth = R_suth .* (V_suth .+ 1.0)

println("Sutherland profile generated in memory at R = ", R_suth, ", Mr(R) = ", Minf_suth * R_suth, ", Tw = ", Tw_suth)
if save_outputs_suth
    println("Target profile: ", profile_R_suth)
    println("All profiles:    ", profile_all_suth)
    println("Summary:         ", summary_suth)
    println("Metadata:        ", metadata_suth)
end

nothing


In [ ]:
# Plot dimensionless profile amplitudes at the specified radius.
plot!(z_suth, U_suth, lw=2, label="U", xlabel="z", ylabel="base-flow amplitude",
     title=@sprintf("Sutherland base flow, Tw=%.3g, Mr=%.3g, R=%.3g", Tw_suth, Mr_suth, R_suth),
     xlims=(0, zmax_suth), legend=:right)
plot!(z_suth, V_suth, lw=2, label="V rotating frame")
plot!(z_suth, W_suth, lw=2, label="W")
plot!(z_suth, T_suth, lw=2, label="T")


In [ ]:
# Plot physical radial and azimuthal velocities at the specified radius.
plot(z_suth, u_physical_suth, lw=2, label="u = R U", xlabel="z", ylabel="physical velocity scale",
     title=@sprintf("Physical velocities at R=%.3g", R_suth), xlims=(0, zmax_suth), legend=:right)
plot!(z_suth, v_rotating_physical_suth, lw=2, label="v_rot = R V")
plot!(z_suth, v_inertial_physical_suth, lw=2, label="v_inertial = R(V+1)")


In [ ]:
# Plot material-property profiles used by the Sutherland model.
plot(z_suth, rho_suth, lw=2, label="rho", xlabel="z", ylabel="property",
     title="Sutherland material-property profiles", xlims=(0, zmax_suth), legend=:right)
plot!(z_suth, mu_suth, lw=2, label="mu")
plot!(z_suth, rho_suth .* mu_suth, lw=2, label="rho*mu")
plot!(z_suth, kappa_suth, lw=2, label="kappa = mu/sigma")
